## ETL Silver – Astronomy (normalización diaria)

### Propósito
Transformar los JSON raw de astronomía (capa Bronze) en una tabla Delta diaria, con 1 fila por `city + date`, deduplicada por `ingestion_time` y con timestamps listos para usar.

### Entrada
- Ruta Bronze:
  - `/Volumes/workspace/default/bronce_clima/astronomy/`
- Formato: JSON con `data` (payload WeatherAPI) + `metadata.ingestion_time`

### Transformaciones principales
- Selección y aplanado de campos desde `data.astronomy.astro`:
  - `sunrise`, `sunset`, `moonrise`, `moonset`, `moon_phase`, `moon_illumination`
- Conversión de `ingestion_time` a timestamp usando el formato:
  - `yyyy-MM-dd'T'HH-mm-ss'Z'`
- Deduplicación:
  - `row_number() over (partition by city, date order by ingestion_time desc)` y se conserva `row_num = 1`
- Construcción de timestamps:
  - `sunrise_ts = to_timestamp(concat(date, sunrise), 'yyyy-MM-dd hh:mm a')`
  - `sunset_ts  = to_timestamp(concat(date, sunset),  'yyyy-MM-dd hh:mm a')`
- Limpieza final:
  - se eliminan `sunrise` y `sunset` (strings) y se conservan `sunrise_ts` y `sunset_ts`

### Salida
- Tabla Delta (managed):
  - `weather_astronomy_daily_silver`
- Granularidad:
  - 1 fila por `city + date`

### Columnas finales (persistidas)
- `city`
- `date`
- `ingestion_time`
- `sunrise_ts`
- `sunset_ts`
- `moonrise`
- `moonset`
- `moon_phase`
- `moon_illumination`


In [0]:
from pyspark.sql.functions import col,explode,lower, regexp_replace,desc,row_number,to_timestamp,to_date,hour,concat_ws
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
df_astronomy_bronze = spark.read.json("/Volumes/workspace/default/bronce_clima/astronomy/")
df_astronomy_bronze.show(5)
df_astronomy_bronze.printSchema()

In [0]:
# 1. Selección de columnas correctas
df_astronomy = df_astronomy_bronze.select(
    col("city"),
    col("date"),
    col("data.astronomy.astro.sunrise").alias("sunrise"),
    col("data.astronomy.astro.sunset").alias("sunset"),
    col("data.astronomy.astro.moonrise").alias("moonrise"),
    col("data.astronomy.astro.moonset").alias("moonset"),
    col("data.astronomy.astro.moon_phase").alias("moon_phase"),
    col("data.astronomy.astro.moon_illumination").alias("moon_illumination"),
    col("metadata.ingestion_time").alias("ingestion_time")
)

# 2. Convertir ingestion_time
df_astronomy = df_astronomy.withColumn(
    "ingestion_time",
    to_timestamp("ingestion_time", "yyyy-MM-dd'T'HH-mm-ss'Z'")
)

# 3. Eliminar duplicados (quedarte con el más reciente)
window_spec = Window.partitionBy("city", "date").orderBy(desc("ingestion_time"))

df_astronomy_latest = (
    df_astronomy
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

In [0]:
df_astronomy_latest = df_astronomy_latest \
    .withColumn("sunrise_ts", to_timestamp(concat_ws(" ", col("date"), col("sunrise")), "yyyy-MM-dd hh:mm a")) \
    .withColumn("sunset_ts", to_timestamp(concat_ws(" ", col("date"), col("sunset")), "yyyy-MM-dd hh:mm a"))
df_astronomy_latest = df_astronomy_latest.drop("sunrise", "sunset")
df_astronomy_latest = df_astronomy_latest.drop("sunrise", "sunset")


In [0]:
df_astronomy_latest.show()
df_astronomy_latest.printSchema()

In [0]:
df_astronomy_latest.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_astronomy_daily_silver")

In [0]:
spark.table("weather_astronomy_daily_silver").show(5)